In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install librosa joblib tqdm imbalanced-learn --quiet

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [4]:
DATA_DIR = "/kaggle/input/datasets/organizations/mozillaorg/common-voice"

AGE_FEATURES_CACHE = "/kaggle/working/age_features_v2.csv"
AGE_MODEL_OUT = "/kaggle/working/age_model.joblib"
AGE_SCALER_OUT = "/kaggle/working/age_scaler.joblib"

N_MFCC = 13
SAMPLE_RATE = 16000
MAX_PER_CLASS = 1500          # cap abundant buckets; rare ones take everything available
CHECKPOINT_EVERY = 500

# Buckets 60+ all collapse into one senior class
SENIOR_SOURCE = {"sixties", "seventies", "eighties", "nineties"}
KEEP_BUCKETS = {"twenties", "thirties", "fourties", "fifties"}

AGE_FEATURE_NAMES = [f"mfcc_{i+1}" for i in range(N_MFCC)] + [
    "f0_mean", "spectral_centroid", "spectral_bandwidth", "zcr"
]

In [5]:
SPLITS = ["cv-valid-train", "cv-valid-dev", "cv-valid-test",
          "cv-other-train", "cv-other-dev", "cv-other-test"]

def load_combined_labeled_df():
    frames = []
    for split in SPLITS:
        csv_path = os.path.join(DATA_DIR, f"{split}.csv")
        if not os.path.exists(csv_path):
            print(f"Skipping {split} (csv not found)")
            continue
        d = pd.read_csv(csv_path)
        d = d[d["gender"].isin(["male", "female"])].dropna(subset=["filename"]).copy()
        d["clips_dir"] = os.path.join(DATA_DIR, split)
        frames.append(d)
        print(f"{split}: {len(d)} labeled rows")
    return pd.concat(frames, ignore_index=True)

combined_df = load_combined_labeled_df()
print("\nTotal labeled rows:", len(combined_df))

cv-valid-train: 73278 labeled rows
cv-valid-dev: 1529 labeled rows
cv-valid-test: 1529 labeled rows
cv-other-train: 63253 labeled rows
cv-other-dev: 1342 labeled rows
cv-other-test: 1272 labeled rows

Total labeled rows: 142203


In [6]:
def to_age_label(bucket):
    if bucket in SENIOR_SOURCE:
        return "senior_60plus"
    if bucket in KEEP_BUCKETS:
        return bucket
    return None  # drop teens (too few + out of your target scheme)

age_df = combined_df[
    (combined_df["gender"] == "male") &
    (combined_df["age"].notna())
].copy()

age_df["age_label"] = age_df["age"].apply(to_age_label)
age_df = age_df[age_df["age_label"].notna()].copy()

print("Available per target class (before sampling):")
print(age_df["age_label"].value_counts())

Available per target class (before sampling):
age_label
twenties         37446
thirties         28440
fourties         16869
fifties           9638
senior_60plus     8513
Name: count, dtype: int64


In [7]:
def build_balanced_sample(df, label_col="age_label", max_per_class=MAX_PER_CLASS):
    parts = []
    for label, grp in df.groupby(label_col):
        n = min(len(grp), max_per_class)
        parts.append(grp.sample(n=n, random_state=42))
    return pd.concat(parts).reset_index(drop=True)

balanced_df = build_balanced_sample(age_df)
print("After balanced sampling:")
print(balanced_df["age_label"].value_counts())

After balanced sampling:
age_label
fifties          1500
fourties         1500
senior_60plus    1500
thirties         1500
twenties         1500
Name: count, dtype: int64


In [8]:
def extract_features(file_path: str):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    if y.size == 0 or np.max(np.abs(y)) < 1e-4:
        return None

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_mean = np.mean(mfcc, axis=1)

    f0, voiced_flag, voiced_probs = librosa.pyin(
        y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
    )
    f0_clean = f0[~np.isnan(f0)]
    f0_mean = np.mean(f0_clean) if f0_clean.size > 0 else 0.0

    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spectral_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    return np.concatenate([mfcc_mean, [f0_mean, spectral_centroid, spectral_bandwidth, zcr]])

In [25]:
def build_age_dataset():
    if os.path.exists(AGE_FEATURES_CACHE):
        print("Loading cached features")
        return pd.read_csv(AGE_FEATURES_CACHE)

    rows = []
    for i, (_, row) in enumerate(tqdm(balanced_df.iterrows(), total=len(balanced_df), desc="Extracting")):
        fpath = os.path.join(row["clips_dir"], row["filename"])
        if not os.path.exists(fpath):
            continue
        feats = extract_features(fpath)
        if feats is None:
            continue
        rows.append(list(feats) + [row["age_label"]])
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows, columns=AGE_FEATURE_NAMES + ["age_label"]).to_csv(AGE_FEATURES_CACHE, index=False)
            print(f"Checkpoint at {i+1}")

    feat_df_age = pd.DataFrame(rows, columns=AGE_FEATURE_NAMES + ["age_label"])
    feat_df_age.to_csv(AGE_FEATURES_CACHE, index=False)
    print(f"Done. {len(feat_df_age)} rows")
    return feat_df_age

feat_df_age = build_age_dataset()
print(feat_df_age["age_label"].value_counts())

Extracting:   7%|▋         | 500/7500 [06:07<1:13:51,  1.58it/s] 

Checkpoint at 500


Extracting:  13%|█▎        | 1000/7500 [11:31<57:40,  1.88it/s] 

Checkpoint at 1000


Extracting:  20%|██        | 1500/7500 [16:57<1:20:50,  1.24it/s]

...

Extracting:  93%|█████████▎| 7000/7500 [1:13:53<05:32,  1.50it/s]

Checkpoint at 7000


Extracting: 100%|██████████| 7500/7500 [1:18:32<00:00,  1.59it/s]

Checkpoint at 7500


Done. 7498 rows
age_label
fifties          1500
twenties         1500
thirties         1500
senior_60plus    1499
fourties         1499
Name: count, dtype: int64


In [26]:
X = feat_df_age[AGE_FEATURE_NAMES].values
y = feat_df_age["age_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

age_scaler = StandardScaler()
X_train_s = age_scaler.fit_transform(X_train)
X_test_s = age_scaler.transform(X_test)

age_clf = RandomForestClassifier(n_estimators=400, max_depth=18, random_state=42, class_weight='balanced')
age_clf.fit(X_train_s, y_train)

age_preds = age_clf.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test, age_preds))
print(classification_report(y_test, age_preds))
print(confusion_matrix(y_test, age_preds, labels=age_clf.classes_))
print("Classes:", age_clf.classes_)

cv = cross_val_score(
    RandomForestClassifier(n_estimators=400, max_depth=18, random_state=42, class_weight='balanced'),
    age_scaler.fit_transform(X), y, cv=5
)
print(f"\nCV accuracy: {cv.mean():.3f} ± {cv.std():.3f}")

Accuracy: 0.6453333333333333
               precision    recall  f1-score   support

      fifties       0.71      0.77      0.74       300
     fourties       0.63      0.69      0.66       300
senior_60plus       0.71      0.81      0.76       300
     thirties       0.56      0.51      0.53       300
     twenties       0.57      0.45      0.50       300

     accuracy                           0.65      1500
    macro avg       0.64      0.65      0.64      1500
 weighted avg       0.64      0.65      0.64      1500

[[232  20  13  15  20]
 [ 19 207  16  35  23]
 [ 12  10 243  17  18]
 [ 33  41  33 152  41]
 [ 31  48  36  51 134]]
Classes: ['fifties' 'fourties' 'senior_60plus' 'thirties' 'twenties']

CV accuracy: 0.633 ± 0.008


In [27]:
print("=== Batch test on 15 real male clips ===")
sample_test = balanced_df.sample(15, random_state=7)

correct = 0
total = 0
for _, row in sample_test.iterrows():
    fpath = os.path.join(row["clips_dir"], row["filename"])
    if not os.path.exists(fpath):
        continue
    feats = extract_features(fpath)
    if feats is None:
        continue
    feats_scaled = age_scaler.transform(feats.reshape(1, -1))
    pred = age_clf.predict(feats_scaled)[0]
    conf = age_clf.predict_proba(feats_scaled).max()
    actual = row["age_label"]
    match = "correct" if pred == actual else "WRONG"
    if pred == actual:
        correct += 1
    total += 1
    print(f"[{match:7s}] actual={actual:14s} predicted={pred:14s} conf={conf*100:.0f}%")

print(f"\nBatch accuracy: {correct}/{total} = {correct/total*100:.0f}%")

=== Batch test on 15 real male clips ===
[correct] actual=fifties        predicted=fifties        conf=86%
[correct] actual=senior_60plus  predicted=senior_60plus  conf=84%
[WRONG  ] actual=twenties       predicted=thirties       conf=26%
[correct] actual=twenties       predicted=twenties       conf=80%
[correct] actual=thirties       predicted=thirties       conf=60%
[correct] actual=fifties        predicted=fifties        conf=73%
[correct] actual=fourties       predicted=fourties       conf=67%
[correct] actual=fourties       predicted=fourties       conf=66%
[correct] actual=thirties       predicted=thirties       conf=66%
[correct] actual=fifties        predicted=fifties        conf=75%
[correct] actual=fourties       predicted=fourties       conf=63%
[correct] actual=fourties       predicted=fourties       conf=90%
[WRONG  ] actual=fourties       predicted=twenties       conf=38%
[correct] actual=fourties       predicted=fourties       conf=88%
[correct] actual=fifties        pre

In [28]:
joblib.dump(age_clf, AGE_MODEL_OUT)
joblib.dump(age_scaler, AGE_SCALER_OUT)
print("Saved age_model.joblib and age_scaler.joblib")

Saved age_model.joblib and age_scaler.joblib


In [29]:
import shutil

senior_samples = combined_df[
    (combined_df["gender"] == "male") &
    (combined_df["age"].isin(["seventies", "eighties"]))
].sample(5, random_state=3)

os.makedirs("/kaggle/working/test_clips", exist_ok=True)
for i, (_, row) in enumerate(senior_samples.iterrows()):
    src = os.path.join(row["clips_dir"], row["filename"])
    if os.path.exists(src):
        dst = f"/kaggle/working/test_clips/senior_{row['age']}_{i}.mp3"
        shutil.copy(src, dst)
        print(f"Saved {dst}")

Saved /kaggle/working/test_clips/senior_seventies_0.mp3
Saved /kaggle/working/test_clips/senior_eighties_1.mp3
Saved /kaggle/working/test_clips/senior_seventies_2.mp3
Saved /kaggle/working/test_clips/senior_seventies_3.mp3
Saved /kaggle/working/test_clips/senior_seventies_4.mp3


In [9]:
import shutil

# Get filenames that WERE used in training, so we can exclude them
trained_filenames = set(balanced_df["filename"].values)

# Pull senior male clips that were NOT in training
unseen_seniors = combined_df[
    (combined_df["gender"] == "male") &
    (combined_df["age"].isin(["seventies", "eighties"])) &
    (~combined_df["filename"].isin(trained_filenames))
].sample(5, random_state=42)

os.makedirs("/kaggle/working/test_clips", exist_ok=True)
for i, (_, row) in enumerate(unseen_seniors.iterrows()):
    src = os.path.join(row["clips_dir"], row["filename"])
    if os.path.exists(src):
        dst = f"/kaggle/working/test_clips/unseen_senior_{row['age']}_{i}.mp3"
        shutil.copy(src, dst)
        print(f"Saved {dst}")

print(f"\nThese {len(unseen_seniors)} clips were NOT used in training.")

Saved /kaggle/working/test_clips/unseen_senior_seventies_0.mp3
Saved /kaggle/working/test_clips/unseen_senior_seventies_1.mp3
Saved /kaggle/working/test_clips/unseen_senior_seventies_2.mp3
Saved /kaggle/working/test_clips/unseen_senior_seventies_3.mp3
Saved /kaggle/working/test_clips/unseen_senior_seventies_4.mp3

These 5 clips were NOT used in training.


In [6]:
import os
CREMA_DIR = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'
files = os.listdir(CREMA_DIR)
print("Total files:", len(files))
print("First 5:", files[:5])

Total files: 7442
First 5: ['1028_TSI_DIS_XX.wav', '1075_IEO_HAP_LO.wav', '1084_ITS_HAP_XX.wav', '1067_IWW_DIS_XX.wav', '1066_TIE_DIS_XX.wav']


In [7]:
CREMA_DIR = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'

EMOTION_MAP = {
    'ANG': 'angry',
    'HAP': 'happy',
    'SAD': 'sad',
    'NEU': 'neutral',
    # dropping DIS (disgust) and FEA (fear) — harder to classify, lower accuracy
}

EMO_FEATURES_CACHE = '/kaggle/working/emotion_features.csv'
EMO_MODEL_OUT = '/kaggle/working/emotion_model.joblib'
EMO_SCALER_OUT = '/kaggle/working/emotion_scaler.joblib'

N_MFCC = 13
SAMPLE_RATE = 16000
CHECKPOINT_EVERY = 500

import os
emotion_files = []
for fname in os.listdir(CREMA_DIR):
    if not fname.endswith('.wav'):
        continue
    parts = fname.split('_')
    if len(parts) < 3:
        continue
    code = parts[2]
    if code in EMOTION_MAP:
        emotion_files.append((fname, EMOTION_MAP[code]))

import pandas as pd
emo_df = pd.DataFrame(emotion_files, columns=['filename', 'emotion'])
print("Total usable files:", len(emo_df))
print(emo_df['emotion'].value_counts())

Total usable files: 4900
emotion
happy      1271
sad        1271
angry      1271
neutral    1087
Name: count, dtype: int64


In [13]:
import numpy as np
import librosa

N_MFCC = 40  # bumped from 13

def extract_emotion_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    if y.size == 0 or np.max(np.abs(y)) < 1e-4:
        return None

    # MFCCs + their deltas (rate of change captures dynamics)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_delta = librosa.feature.delta(mfcc)

    # mean AND std for each — std captures emotional variation over time
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    mfcc_delta_mean = np.mean(mfcc_delta, axis=1)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    chroma_std = np.std(chroma, axis=1)

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=13)
    mel_mean = np.mean(mel, axis=1)

    # Prosodic / spectral extras (mean + std where it helps)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    extras = [
        np.mean(zcr), np.std(zcr),
        np.mean(rms), np.std(rms),
        np.mean(spectral_centroid), np.std(spectral_centroid),
    ]

    return np.concatenate([
        mfcc_mean, mfcc_std, mfcc_delta_mean,
        chroma_mean, chroma_std,
        mel_mean,
        extras
    ])

EMO_FEATURE_NAMES = (
    [f"mfcc_mean_{i+1}" for i in range(N_MFCC)] +
    [f"mfcc_std_{i+1}" for i in range(N_MFCC)] +
    [f"mfcc_delta_{i+1}" for i in range(N_MFCC)] +
    [f"chroma_mean_{i+1}" for i in range(12)] +
    [f"chroma_std_{i+1}" for i in range(12)] +
    [f"mel_{i+1}" for i in range(13)] +
    ["zcr_mean", "zcr_std", "rms_mean", "rms_std", "centroid_mean", "centroid_std"]
)
print("Total features:", len(EMO_FEATURE_NAMES))

Total features: 163


In [15]:
from tqdm import tqdm
if os.path.exists(EMO_FEATURES_CACHE):
    os.remove(EMO_FEATURES_CACHE)
def build_emotion_dataset():
    if os.path.exists(EMO_FEATURES_CACHE):
        print("Loading cached emotion features")
        return pd.read_csv(EMO_FEATURES_CACHE)

    rows = []
    for i, (_, row) in enumerate(tqdm(emo_df.iterrows(), total=len(emo_df), desc="Extracting emotion features")):
        fpath = os.path.join(CREMA_DIR, row["filename"])
        if not os.path.exists(fpath):
            continue
        feats = extract_emotion_features(fpath)
        if feats is None:
            continue
        rows.append(list(feats) + [row["emotion"]])
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows, columns=EMO_FEATURE_NAMES + ["emotion"]).to_csv(EMO_FEATURES_CACHE, index=False)
            print(f"Checkpoint at {i+1}")

    feat_df_emo = pd.DataFrame(rows, columns=EMO_FEATURE_NAMES + ["emotion"])
    feat_df_emo.to_csv(EMO_FEATURES_CACHE, index=False)
    print(f"Done. {len(feat_df_emo)} rows")
    return feat_df_emo

feat_df_emo = build_emotion_dataset()
print(feat_df_emo["emotion"].value_counts())

Extracting emotion features:  10%|█         | 505/4900 [00:13<02:51, 25.59it/s]

Checkpoint at 500


Extracting emotion features:  21%|██        | 1007/4900 [00:27<03:06, 20.92it/s]

Checkpoint at 1000


Extracting emotion features:  31%|███       | 1504/4900 [00:41<03:08, 18.04it/s]

...

Checkpoint at 4000


Extracting emotion features:  92%|█████████▏| 4503/4900 [02:06<00:57,  6.93it/s]

Checkpoint at 4500


Extracting emotion features: 100%|██████████| 4900/4900 [02:17<00:00, 35.69it/s]


Done. 4899 rows
emotion
happy      1271
angry      1271
sad        1270
neutral    1087
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

X = feat_df_emo[EMO_FEATURE_NAMES].values
y = feat_df_emo["emotion"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

emo_scaler = StandardScaler()
X_train_s = emo_scaler.fit_transform(X_train)
X_test_s = emo_scaler.transform(X_test)

emo_clf = RandomForestClassifier(n_estimators=400, max_depth=20, random_state=42, class_weight='balanced')
emo_clf.fit(X_train_s, y_train)

emo_preds = emo_clf.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test, emo_preds))
print(classification_report(y_test, emo_preds))
print(confusion_matrix(y_test, emo_preds, labels=emo_clf.classes_))
print("Classes:", emo_clf.classes_)

cv = cross_val_score(
    RandomForestClassifier(n_estimators=400, max_depth=20, random_state=42, class_weight='balanced'),
    emo_scaler.fit_transform(X), y, cv=5
)
print(f"\nCV accuracy: {cv.mean():.3f} ± {cv.std():.3f}")

Accuracy: 0.713265306122449
              precision    recall  f1-score   support

       angry       0.79      0.81      0.80       254
       happy       0.69      0.56      0.62       254
     neutral       0.59      0.71      0.64       218
         sad       0.79      0.78      0.78       254

    accuracy                           0.71       980
   macro avg       0.71      0.71      0.71       980
weighted avg       0.72      0.71      0.71       980

[[205  34  13   2]
 [ 49 142  50  13]
 [  2  23 154  39]
 [  2   8  46 198]]
Classes: ['angry' 'happy' 'neutral' 'sad']

CV accuracy: 0.680 ± 0.008


In [22]:
CREMA_DIR = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'

EMOTION_MAP = {'ANG': 'angry', 'HAP': 'happy', 'SAD': 'sad'}

EMO_FEATURES_CACHE = '/kaggle/working/emotion_features_v2.csv'
EMO_MODEL_OUT = '/kaggle/working/emotion_model.joblib'
EMO_SCALER_OUT = '/kaggle/working/emotion_scaler.joblib'

N_MFCC = 40
SAMPLE_RATE = 16000
CHECKPOINT_EVERY = 500

import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

emotion_files = []
for fname in os.listdir(CREMA_DIR):
    if not fname.endswith('.wav'):
        continue
    parts = fname.split('_')
    if len(parts) >= 3 and parts[2] in EMOTION_MAP:
        emotion_files.append((fname, EMOTION_MAP[parts[2]]))

emo_df = pd.DataFrame(emotion_files, columns=['filename', 'emotion'])
print("Total usable files:", len(emo_df))
print(emo_df['emotion'].value_counts())

Total usable files: 3813
emotion
happy    1271
sad      1271
angry    1271
Name: count, dtype: int64


In [23]:
def extract_from_array(y, sr):
    if y.size == 0 or np.max(np.abs(y)) < 1e-4:
        return None
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_mean = np.mean(mfcc, axis=1); mfcc_std = np.std(mfcc, axis=1)
    mfcc_delta_mean = np.mean(mfcc_delta, axis=1)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1); chroma_std = np.std(chroma, axis=1)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=13)
    mel_mean = np.mean(mel, axis=1)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    sc = librosa.feature.spectral_centroid(y=y, sr=sr)
    extras = [np.mean(zcr), np.std(zcr), np.mean(rms), np.std(rms), np.mean(sc), np.std(sc)]
    return np.concatenate([mfcc_mean, mfcc_std, mfcc_delta_mean, chroma_mean, chroma_std, mel_mean, extras])

def extract_emotion_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    return extract_from_array(y, sr)

EMO_FEATURE_NAMES = (
    [f"mfcc_mean_{i+1}" for i in range(N_MFCC)] +
    [f"mfcc_std_{i+1}" for i in range(N_MFCC)] +
    [f"mfcc_delta_{i+1}" for i in range(N_MFCC)] +
    [f"chroma_mean_{i+1}" for i in range(12)] +
    [f"chroma_std_{i+1}" for i in range(12)] +
    [f"mel_{i+1}" for i in range(13)] +
    ["zcr_mean", "zcr_std", "rms_mean", "rms_std", "centroid_mean", "centroid_std"]
)
print("Total features:", len(EMO_FEATURE_NAMES))

Total features: 163


In [24]:
def build_emotion_dataset():
    if os.path.exists(EMO_FEATURES_CACHE):
        print("Loading cached")
        return pd.read_csv(EMO_FEATURES_CACHE)

    rows = []
    for i, (_, row) in enumerate(tqdm(emo_df.iterrows(), total=len(emo_df), desc="Extracting")):
        fpath = os.path.join(CREMA_DIR, row["filename"])
        if not os.path.exists(fpath):
            continue
        feats = extract_emotion_features(fpath)
        if feats is None:
            continue
        rows.append(list(feats) + [row["emotion"], row["filename"]])
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows, columns=EMO_FEATURE_NAMES + ["emotion", "filename"]).to_csv(EMO_FEATURES_CACHE, index=False)
            print(f"Checkpoint {i+1}")

    out = pd.DataFrame(rows, columns=EMO_FEATURE_NAMES + ["emotion", "filename"])
    out.to_csv(EMO_FEATURES_CACHE, index=False)
    print(f"Done. {len(out)} rows")
    return out

if os.path.exists(EMO_FEATURES_CACHE):
    os.remove(EMO_FEATURES_CACHE)

feat_df_emo = build_emotion_dataset()
print(feat_df_emo["emotion"].value_counts())

Extracting:  13%|█▎        | 504/3813 [00:13<02:19, 23.78it/s]

Checkpoint 500


Extracting:  26%|██▋       | 1005/3813 [00:26<02:15, 20.74it/s]

Checkpoint 1000


Extracting:  39%|███▉      | 1504/3813 [00:40<02:15, 17.03it/s]

...

Checkpoint 3000


Extracting:  92%|█████████▏| 3507/3813 [01:36<00:26, 11.39it/s]

Checkpoint 3500


Extracting: 100%|██████████| 3813/3813 [01:44<00:00, 36.59it/s]


Done. 3812 rows
emotion
happy    1271
angry    1271
sad      1270
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

X = feat_df_emo[EMO_FEATURE_NAMES].values
y = feat_df_emo["emotion"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

emo_scaler = StandardScaler()
X_train_s = emo_scaler.fit_transform(X_train)
X_test_s = emo_scaler.transform(X_test)

emo_clf = RandomForestClassifier(n_estimators=400, max_depth=20, random_state=42, class_weight='balanced')
emo_clf.fit(X_train_s, y_train)

emo_preds = emo_clf.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test, emo_preds))
print(classification_report(y_test, emo_preds))
print(confusion_matrix(y_test, emo_preds, labels=emo_clf.classes_))

joblib.dump(emo_clf, EMO_MODEL_OUT)
joblib.dump(emo_scaler, EMO_SCALER_OUT)
print("Saved emotion model + scaler")

Accuracy: 0.7745740498034076
              precision    recall  f1-score   support

       angry       0.77      0.79      0.78       255
       happy       0.68      0.65      0.67       254
         sad       0.87      0.88      0.87       254

    accuracy                           0.77       763
   macro avg       0.77      0.77      0.77       763
weighted avg       0.77      0.77      0.77       763

[[202  50   3]
 [ 57 166  31]
 [  2  29 223]]
Saved emotion model + scaler


In [26]:
import random

# Pull random clips from the dataset (across all 3 emotions)
test_sample = emo_df.sample(15, random_state=99)

print("=== Emotion model — test on 15 random clips ===\n")
correct = 0
total = 0
for _, row in test_sample.iterrows():
    fpath = os.path.join(CREMA_DIR, row["filename"])
    if not os.path.exists(fpath):
        continue
    feats = extract_emotion_features(fpath)
    if feats is None:
        continue
    feats_scaled = emo_scaler.transform(feats.reshape(1, -1))
    pred = emo_clf.predict(feats_scaled)[0]
    conf = emo_clf.predict_proba(feats_scaled).max()
    actual = row["emotion"]
    match = "correct" if pred == actual else "WRONG"
    if pred == actual:
        correct += 1
    total += 1
    print(f"[{match:7s}] actual={actual:8s} predicted={pred:8s} conf={conf*100:.0f}%  ({row['filename']})")

print(f"\nBatch: {correct}/{total} correct = {correct/total*100:.0f}%")

=== Emotion model — test on 15 random clips ===

[correct] actual=angry    predicted=angry    conf=88%  (1022_IWL_ANG_XX.wav)
[correct] actual=happy    predicted=happy    conf=78%  (1085_TSI_HAP_XX.wav)
[correct] actual=happy    predicted=happy    conf=70%  (1032_DFA_HAP_XX.wav)
[correct] actual=sad      predicted=sad      conf=73%  (1011_IEO_SAD_LO.wav)
[correct] actual=angry    predicted=angry    conf=91%  (1039_TAI_ANG_XX.wav)
[correct] actual=happy    predicted=happy    conf=66%  (1059_ITS_HAP_XX.wav)
[correct] actual=sad      predicted=sad      conf=99%  (1007_IWW_SAD_XX.wav)
[correct] actual=sad      predicted=sad      conf=99%  (1047_TAI_SAD_XX.wav)
[correct] actual=sad      predicted=sad      conf=99%  (1033_IEO_SAD_LO.wav)
[correct] actual=angry    predicted=angry    conf=95%  (1059_DFA_ANG_XX.wav)
[correct] actual=happy    predicted=happy    conf=75%  (1089_TIE_HAP_XX.wav)
[correct] actual=angry    predicted=angry    conf=81%  (1055_MTI_ANG_XX.wav)
[correct] actual=angry    p

In [27]:
import shutil

# Age model uses the 30-feature extraction; emotion data is CREMA-D clips.
# We find CREMA-D clips that YOUR age model flags as senior, so the GUI emotion branch fires.

def age_extract(file_path):
    """30-feature extraction matching the AGE model (not the emotion one)."""
    y, sr = librosa.load(file_path, sr=16000)
    if y.size == 0 or np.max(np.abs(y)) < 1e-4:
        return None
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"))
    f0_clean = f0[~np.isnan(f0)]
    f0_mean = np.mean(f0_clean) if f0_clean.size > 0 else 0.0
    sc = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    sb = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    return np.concatenate([mfcc_mean, [f0_mean, sc, sb, zcr]])

os.makedirs("/kaggle/working/senior_demo_clips", exist_ok=True)

found = 0
# scan a sample of CREMA-D clips, keep ones the age model calls senior
for _, row in emo_df.sample(200, random_state=3).iterrows():
    fpath = os.path.join(CREMA_DIR, row["filename"])
    if not os.path.exists(fpath):
        continue
    feats = age_extract(fpath)
    if feats is None:
        continue
    pred = age_clf.predict(age_scaler.transform(feats.reshape(1, -1)))[0]
    if pred == "senior_60plus":
        dst = f"/kaggle/working/senior_demo_clips/senior_{row['emotion']}_{found}.wav"
        shutil.copy(fpath, dst)
        print(f"Saved {dst}  (true emotion: {row['emotion']})")
        found += 1
    if found >= 6:
        break

if found == 0:
    print("No clips flagged senior in this sample — try increasing the sample size from 200.")
else:
    print(f"\nFound {found}. Download these from the Output tab and upload to the GUI to see the full pipeline.")

NameError: name 'age_clf' is not defined